In [5]:
!pip install lightgbm

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)


# 유방암 분류 모델 개선
XGBoost 튜닝, RandomizedSearchCV, Optuna 모델 비교

> 이 노트북은 강의 슬라이드(PDF)에 포함된 코드만 순서대로 정리한 것입니다.

## 1. 라이브러리 불러오기

In [1]:
!pip install xgboost lightgbm optuna

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
  Using cached optuna-4.9.0-py3-none-any.whl.metadata (15 kB)
  Using cached alembic-1.16.5-py3-none-any.whl.metadata (7.3 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached sqlalchemy-2.0.51-cp39-cp39-win_amd64.whl.metadata (9.8 kB)
  Using cached mako-1.3.12-py3-none-any.whl.metadata (2.9 kB)
  Using cached greenlet-3.2.5.tar.gz (191 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
Using cached optuna-4.9.0-py3-none-any.whl (425 kB)
Using cached alembic-1.16.5-py3-none-any.whl (247 kB)
Using cached sqlalchemy-2.0.51-cp39-cp39-win_amd64.whl (2.1

  error: subprocess-exited-with-error
  
  × Building wheel for greenlet (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [112 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-39\greenlet
      copying src\greenlet\__init__.py -> build\lib.win-amd64-cpython-39\greenlet
      creating build\lib.win-amd64-cpython-39\greenlet\platform
      copying src\greenlet\platform\__init__.py -> build\lib.win-amd64-cpython-39\greenlet\platform
      creating build\lib.win-amd64-cpython-39\greenlet\tests
      copying src\greenlet\tests\fail_clearing_run_switches.py -> build\lib.win-amd64-cpython-39\greenlet\tests
      copying src\greenlet\tests\fail_cpp_exception.py -> build\lib.win-amd64-cpython-39\greenlet\tests
      copying src\greenlet\tests\fail_initialstub_already_started.py -> build\lib.win-amd64-cpython-39\greenlet\tests
      copying src\greenlet\tests\fail_slp_switch.py -> build\lib.win-amd64

In [2]:
# 숫자 계산과 표 데이터 처리를 위한 기본 라이브러리
import numpy as np              # 숫자 계산을 도와주는 기본 라이브러리
import pandas as pd             # 표(DataFrame) 형태 데이터를 다루기 위해 사용
import matplotlib.pyplot as plt # 그래프를 그리기 위해 사용

# 예제 데이터셋(유방암 데이터)을 불러오는 도구
from sklearn.datasets import load_breast_cancer

# 데이터를 학습용/시험용으로 나누고, 교차검증을 하기 위한 도구들
from sklearn.model_selection import train_test_split       # 데이터를 train(공부용)/test(시험용)로 나눔
from sklearn.model_selection import StratifiedKFold        # 교차검증 시 benign/malignant 비율을 유지하며 데이터를 나눔
from sklearn.model_selection import RandomizedSearchCV     # 하이퍼파라미터 조합을 '무작위로' 골라 실험하는 튜닝 도구
from sklearn.model_selection import cross_val_score        # 교차검증 점수를 계산 (뒤에서 Optuna 함수 안에서 사용)

In [3]:
# feature 값의 크기를 비슷하게 맞추는 도구와, 전처리+모델을 묶는 도구
from sklearn.preprocessing import StandardScaler   # feature 값을 비슷한 범위로 맞춤(스케일 조정)
from sklearn.pipeline import Pipeline               # 전처리와 모델을 순서대로 묶어 한 번에 실행

# 모델 성능을 평가하는 여러 지표들
from sklearn.metrics import accuracy_score            # 전체 정답률
from sklearn.metrics import precision_score            # 정밀도
from sklearn.metrics import recall_score                # 재현율 (악성을 놓치지 않는 능력)
from sklearn.metrics import f1_score                     # precision과 recall의 균형 점수
from sklearn.metrics import roc_auc_score                 # 확률 기준 전반적 구분 능력(ROC-AUC)
from sklearn.metrics import confusion_matrix                # 정답/오답 유형을 정리한 표(혼동 행렬)
from sklearn.metrics import classification_report            # 클래스별 precision/recall 요약 출력
from sklearn.metrics import ConfusionMatrixDisplay             # 혼동 행렬을 '그림'으로 그려주는 도구
from sklearn.metrics import RocCurveDisplay                     # ROC Curve를 '그림'으로 그려주는 도구

In [4]:
# 비교에 사용할 여러 분류 모델들
from sklearn.linear_model import LogisticRegression    # 기본적인 분류 모델
from sklearn.ensemble import RandomForestClassifier     # 여러 Decision Tree를 종합 판단하는 모델
from xgboost import XGBClassifier                        # 고성능 부스팅 모델 (이번 수업의 주인공)
from lightgbm import LGBMClassifier                        # XGBoost와 비슷한 또 다른 고성능 부스팅 모델

# 하이퍼파라미터 자동 튜닝 도구
import optuna   # 이전 실험 결과를 참고해 다음 설정을 똑똑하게 탐색

print("라이브러리 불러오기 완료")

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
# ------------------------------------------------------------
# Windows 한글 폰트 설정
# ------------------------------------------------------------
# matplotlib의 기본 글꼴은 DejaVu Sans인데, 이 글꼴은 한글을 제대로 표시하지 못합니다.
# 그래서 Windows에 기본으로 설치된 '맑은 고딕'을 그래프 글꼴로 지정합니다.
plt.rcParams["font.family"] = "Malgun Gothic"

# 마이너스(-) 기호가 깨지는 문제를 방지합니다.
# 예: -10 같은 값이 □10 처럼 보이는 문제 예방
plt.rcParams["axes.unicode_minus"] = False

## 2. 데이터 준비
유방암 데이터 · target 변환 · X/y 분리 · train/test 분리

In [ ]:
# 유방암 예제 데이터를 불러옵니다.
data = load_breast_cancer()

# data.data           : feature, 즉 모델이 입력으로 사용할 숫자 정보들
# data.feature_names   : 각 feature의 이름
# 이 둘을 합쳐서 표(DataFrame) 형태로 만듭니다.
df = pd.DataFrame(data.data, columns=data.feature_names)

In [ ]:
# ----- 원본 target 보존 -----
# sklearn 원본 기준:
# 0 = malignant, 악성
# 1 = benign, 양성
#
# 원본 target을 보존해 두면,
# 나중에 "원래 데이터에서는 어떤 기준이었는지" 확인할 수 있습니다.
df["target_original"] = data.target

In [ ]:
# ----- 수업용 target 생성 -----
# 원본 target의 0과 1을 서로 뒤집습니다.
#
# 계산 방식:
# 원본이 0이면 1 - 0 = 1
# 원본이 1이면 1 - 1 = 0
#
# 수업용 기준:
# 0 = benign, 양성
# 1 = malignant, 악성
#
# 이렇게 바꾸는 이유:
# 이번 수업에서는 악성(malignant)을 놓치는 FN이 중요하므로,
# 악성을 관심 클래스인 1로 두면 recall, FN, TP 해석이 쉬워집니다.
df["target"] = 1 - df["target_original"]

In [ ]:
# ----- 사람이 읽기 쉬운 이름 컬럼 추가 -----
# 숫자 0/1만 보면 의미가 바로 떠오르지 않을 수 있습니다.
# 그래서 target_name 컬럼을 추가해 benign/malignant 이름을 함께 확인합니다.
df["target_name"] = df["target"].map({
    0: "benign",
    1: "malignant"
})

In [ ]:
# 표의 크기, 즉 행 개수와 열 개수를 확인합니다.
print("데이터 크기 (행, 열):", df.shape)

# target 변환이 잘 되었는지 확인합니다.
print("\n원본 target 기준 개수:")
print(df["target_original"].value_counts().sort_index())

print("\n수업용 target 기준 개수:")
print(df["target"].value_counts().sort_index())

print("\n수업용 target 이름 기준 개수:")
print(df["target_name"].value_counts())

# 예상 출력:
# 데이터 크기 (행, 열): (569, 33)
#
# 원본 target 기준 개수:
# target_original
# 0    212
# 1    357
# Name: count, dtype: int64
#
# 수업용 target 기준 개수:
# target
# 0    357
# 1    212
# Name: count, dtype: int64
#
# 수업용 target 이름 기준 개수:
# target_name
# benign       357
# malignant    212
# Name: count, dtype: int64

데이터 크기 (행, 열): (569, 33)

원본 target 기준 개수:
target_original
0    212
1    357
Name: count, dtype: int64

수업용 target 기준 개수:
target
0    357
1    212
Name: count, dtype: int64

수업용 target 이름 기준 개수:
target_name
benign       357
malignant    212
Name: count, dtype: int64


In [ ]:
# 모델 입력값 X를 만듭니다.
# target_original, target, target_name은 정답 관련 컬럼이므로 feature에서 제외합니다.
X = df.drop(columns=["target_original", "target", "target_name"])

# 모델이 맞혀야 하는 정답 y를 만듭니다.
# 수업용 기준:
# 0 = benign
# 1 = malignant
y = df["target"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)

# 예상 출력:
# X 크기: (569, 30)
# y 크기: (569,)

X 크기: (569, 30)
y 크기: (569,)


In [ ]:
# 데이터를 학습용(train) 80%, 시험용(test) 20%로 나눕니다.
# random_state=42 : 나누는 결과를 매번 똑같이 재현하기 위한 고정값
# stratify=y       : benign/malignant 비율을 train과 test에 비슷하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train 크기:", X_train.shape)
print("test 크기:", X_test.shape)
print("\ntrain의 target 비율:")
print(y_train.value_counts(normalize=True).round(3))
print("\ntest의 target 비율:")
print(y_test.value_counts(normalize=True).round(3))

# 예상 출력:
# train 크기: (455, 30)
# test 크기: (114, 30)
#
# train의 target 비율:
# target
# 0    0.626
# 1    0.374
# Name: proportion, dtype: float64
#
# test의 target 비율:
# target
# 0    0.632
# 1    0.368
# Name: proportion, dtype: float64

train 크기: (455, 30)
test 크기: (114, 30)

train의 target 비율:
target
0    0.626
1    0.374
Name: proportion, dtype: float64

test의 target 비율:
target
0    0.632
1    0.368
Name: proportion, dtype: float64


## 3. 공통 평가 함수
공통 평가 지표 · precision/recall · FN · ROC-AUC

In [ ]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    """
    분류 모델의 성능을 같은 기준으로 평가하는 함수입니다.

    이 수업에서는 target을 다음 기준으로 사용합니다.

    0 = benign, 양성
    1 = malignant, 악성

    따라서 precision, recall, f1-score는
    malignant, 즉 1번 클래스를 기준으로 계산합니다.

    model_name : 모델 이름 (비교표에서 구분용)
    y_true     : 실제 정답
    y_pred     : 모델이 예측한 0/1 값
    y_proba    : malignant(=1)일 확률 (predict_proba(X)[:, 1])
    """

    # 전체 중 맞춘 비율 (단, accuracy 하나만 보면 안 됩니다)
    accuracy = accuracy_score(y_true, y_pred)

    # malignant(=1)로 예측한 것 중 실제로 malignant인 비율
    precision_malignant = precision_score(y_true, y_pred, pos_label=1)

    # 실제 malignant 중 모델이 malignant로 맞춘 비율 (놓치지 않는 능력)
    recall_malignant = recall_score(y_true, y_pred, pos_label=1)

    # precision과 recall의 균형 점수
    f1_malignant = f1_score(y_true, y_pred, pos_label=1)

    # 확률 기준 전반적인 구분 능력 (threshold 하나에 묶이지 않음)
    roc_auc = roc_auc_score(y_true, y_proba)

    # confusion_matrix는 기본적으로 아래 순서로 결과를 반환합니다.
    # [[TN, FP],
    #  [FN, TP]]
    #
    # TN: 실제 benign이고 benign으로 예측한 경우
    # FP: 실제 benign인데 malignant로 예측한 경우
    # FN: 실제 malignant인데 benign으로 예측한 경우  <- 의료에서 특히 위험
    # TP: 실제 malignant이고 malignant로 예측한 경우
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    result = {
        "model_name": model_name,
        "accuracy": accuracy,
        "precision_malignant": precision_malignant,
        "recall_malignant": recall_malignant,
        "f1_malignant": f1_malignant,
        "roc_auc": roc_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    return result

print("평가 함수 준비 완료")

평가 함수 준비 완료


## 4. XGBoost 튜닝 실험
Base 모델 · 하이퍼파라미터 · RandomizedSearchCV · Optuna

### 4-1. XGBoost Base 모델

In [ ]:
# 기본 설정값으로 XGBoost 모델을 만듭니다. (아직 튜닝 안 함)
xgb_base_model = XGBClassifier(
    n_estimators=100,      # 트리 100개
    max_depth=3,            # 트리 깊이 3
    learning_rate=0.1,       # 학습 속도(보폭)
    random_state=42,          # 재현성 고정
    eval_metric="logloss"      # 학습 중 사용할 평가 기준
)

In [ ]:
# train 데이터로 학습합니다.
xgb_base_model.fit(X_train, y_train)

# test 데이터로 예측합니다.
# predict()는 최종적으로 0(benign) 또는 1(malignant)을 '정해서' 돌려줍니다.
xgb_base_pred = xgb_base_model.predict(X_test)

# predict_proba()는 0/1로 딱 정하기 전의 '확률'을 돌려줍니다. (예: malignant일 확률 0.83)
# 여기서는 malignant(=1)일 확률만 따로 뽑습니다.
# ROC-AUC를 계산하려면 이 확률값이 필요합니다. (확률이 있어야 여러 기준선으로 평가 가능)
xgb_base_proba = xgb_base_model.predict_proba(X_test)[:, 1]

In [ ]:
# 공통 평가 함수로 성능을 정리합니다.
xgb_base_result = evaluate_classification_model(
    model_name="XGBoost Base",
    y_true=y_test,
    y_pred=xgb_base_pred,
    y_proba=xgb_base_proba
)

xgb_base_result

# 예상 출력:
# {'model_name': 'XGBoost Base',
#  'accuracy': 0.9649122807017544,
#  'precision_malignant': 1.0,
#  'recall_malignant': 0.9047619047619048,
#  'f1_malignant': 0.95,
#  'roc_auc': 0.9966931216931217,
#  'TN': np.int64(72),
#  'FP': np.int64(0),
#  'FN': np.int64(4),
#  'TP': np.int64(38)}

{'model_name': 'XGBoost Base',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9943783068783068,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [ ]:
# 더 자세한 평가 결과도 함께 살펴봅니다.
# target_names를 지정해 benign/malignant 이름으로 보기 쉽게 출력합니다.
print("=== Classification Report (XGBoost Base) ===")
print(classification_report(
    y_test,
    xgb_base_pred,
    target_names=["benign(0)", "malignant(1)"]
))

print("=== Confusion Matrix (XGBoost Base) ===")
print(confusion_matrix(y_test, xgb_base_pred))
print("\n[[TN, FP],")
print(" [FN, TP]] 순서입니다.")

=== Classification Report (XGBoost Base) ===
              precision    recall  f1-score   support

   benign(0)       0.95      1.00      0.97        72
malignant(1)       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

=== Confusion Matrix (XGBoost Base) ===
[[72  0]
 [ 4 38]]

[[TN, FP],
 [FN, TP]] 순서입니다.


### 4-2. RandomizedSearchCV

In [ ]:
# 튜닝에 사용할 XGBoost (설정값은 비워두고 탐색으로 채울 예정)
xgb_for_random_search = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

# 탐색할 하이퍼파라미터 후보 (수업 시간을 고려해 너무 넓지 않게 제한)
param_distributions = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0]
}

# 교차검증 방식: 3등분, 섞어서 나눔, 재현성 고정
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("탐색 범위 설정 완료")

탐색 범위 설정 완료


In [ ]:
# RandomizedSearchCV 객체를 만듭니다.
random_search = RandomizedSearchCV(
    estimator=xgb_for_random_search,
    param_distributions=param_distributions,
    n_iter=20,                # 무작위 조합 20개
    scoring="roc_auc",         # ROC-AUC 기준으로 좋은 조합 찾기
    cv=cv,                      # 위에서 만든 3등분 교차검증
    random_state=42,
    n_jobs=-1                    # CPU를 최대한 활용
)

# train 데이터 안에서 탐색을 실행합니다. (test는 사용하지 않음)
random_search.fit(X_train, y_train)

print("RandomizedSearchCV 탐색 완료")

RandomizedSearchCV 탐색 완료


In [ ]:
# 탐색 결과를 확인합니다.
print("Best ROC-AUC:", random_search.best_score_)
print("Best Params:", random_search.best_params_)
print("Best Estimator:", random_search.best_estimator_)

Best ROC-AUC: 0.9899914259332542
Best Params: {'subsample': 0.7, 'reg_lambda': 2.0, 'reg_alpha': 1.0, 'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
Best Estimator: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, ...)


In [ ]:
# 탐색으로 찾은 가장 좋은 모델을 꺼냅니다.
xgb_random_search_model = random_search.best_estimator_

# test 데이터로 예측
xgb_random_search_pred = xgb_random_search_model.predict(X_test)
xgb_random_search_proba = xgb_random_search_model.predict_proba(X_test)[:, 1]

# 공통 평가 함수로 정리
xgb_random_search_result = evaluate_classification_model(
    model_name="XGBoost RandomizedSearchCV",
    y_true=y_test,
    y_pred=xgb_random_search_pred,
    y_proba=xgb_random_search_proba
)

xgb_random_search_result

# 예상 출력:
# {'model_name': 'XGBoost RandomizedSearchCV',
#  'accuracy': 0.9824561403508771,
#  'precision_malignant': 1.0,
#  'recall_malignant': 0.9523809523809523,
#  'f1_malignant': 0.975609756097561,
#  'roc_auc': 0.9930555555555555,
#  'TN': np.int64(72),
#  'FP': np.int64(0),
#  'FN': np.int64(2),
#  'TP': np.int64(40)}

{'model_name': 'XGBoost RandomizedSearchCV',
 'accuracy': 0.9736842105263158,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9629629629629629,
 'roc_auc': 0.9940476190476191,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(3),
 'TP': np.int64(39)}

### 4-3. Optuna 튜닝

In [ ]:
def objective(trial):
    """
    Optuna가 XGBoost 하이퍼파라미터를 실험할 때 사용할 함수입니다.

    trial은 한 번의 실험을 의미합니다.
    Optuna는 trial마다 서로 다른 하이퍼파라미터 값을 넣어보고,
    그 결과 점수가 좋은지 확인합니다.
    """

    # trial.suggest_* : Optuna가 정해진 범위 안에서 값을 하나 골라줍니다.
    params = {
        # 트리 개수: 50~300 사이 정수 중에서 선택
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        # 트리 깊이: 2~6 사이 정수
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        # 학습 속도: 0.01~0.2, log=True는 작은 값 쪽을 더 촘촘히 탐색
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        # 행 샘플링 비율: 0.7~1.0
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        # feature 샘플링 비율: 0.7~1.0
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        # L1 규제: 0.0~1.0
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        # L2 규제: 0.5~3.0
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
        # 재현성 고정
        "random_state": 42,
        "eval_metric": "logloss"
    }

    # 위에서 고른 설정값으로 XGBoost 모델을 만듭니다.
    model = XGBClassifier(**params)

    # train 데이터 안에서 3등분 교차검증
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    # 교차검증으로 ROC-AUC 점수들을 계산합니다. (test는 사용 안 함)
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    # 교차검증 점수들의 평균을 이 trial의 점수로 돌려줍니다.
    return scores.mean()

print("objective 함수 준비 완료")

objective 함수 준비 완료


In [ ]:
# 로그가 너무 많이 출력되면 수업 집중이 어려우므로 경고 수준만 보이도록 낮춥니다.
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 재현성을 위한 샘플러
sampler = optuna.samplers.TPESampler(seed=42)

# 점수를 최대화하는 study 생성
study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# objective 함수를 20번 실험합니다.
study.optimize(objective, n_trials=20)

print("Optuna 튜닝 완료")

Optuna 튜닝 완료


In [ ]:
# 가장 좋았던 점수와 그 점수를 만든 설정값
print("Best ROC-AUC:", study.best_value)
print("Best Params:", study.best_params)

# 각 trial의 기록을 표로 확인 (상위 일부만)
study.trials_dataframe().head()

Best ROC-AUC: 0.9914808952205073
Best Params: {'n_estimators': 167, 'max_depth': 2, 'learning_rate': 0.06808451320965114, 'subsample': 0.7349415866941288, 'colsample_bytree': 0.7410426998135636, 'reg_alpha': 0.6242658892594297, 'reg_lambda': 2.187097149005392}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.989862,2026-07-01 09:05:23.319756,2026-07-01 09:05:23.539040,0 days 00:00:00.219284,0.746806,0.089608,6,144,0.155995,0.645209,0.879598,COMPLETE
1,1,0.990551,2026-07-01 09:05:23.539116,2026-07-01 09:05:23.909834,0 days 00:00:00.370718,0.990973,0.083411,5,267,0.832443,1.030848,0.706175,COMPLETE
2,2,0.985087,2026-07-01 09:05:23.909895,2026-07-01 09:05:24.048703,0 days 00:00:00.138808,0.829584,0.024879,2,95,0.291229,2.029632,0.857427,COMPLETE
3,3,0.985759,2026-07-01 09:05:24.048762,2026-07-01 09:05:24.246851,0 days 00:00:00.198089,0.935553,0.029967,3,85,0.199674,1.785586,0.836821,COMPLETE
4,4,0.990731,2026-07-01 09:05:24.246904,2026-07-01 09:05:24.451712,0 days 00:00:00.204808,0.719515,0.061721,2,198,0.948886,2.914080,0.751157,COMPLETE


In [ ]:
# best_params에 재현성/평가 설정을 더해 최종 모델을 만듭니다.
xgb_optuna_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss"
)

# train 전체로 다시 학습
xgb_optuna_model.fit(X_train, y_train)

# test로 예측
xgb_optuna_pred = xgb_optuna_model.predict(X_test)
xgb_optuna_proba = xgb_optuna_model.predict_proba(X_test)[:, 1]

# 공통 평가 함수로 정리
xgb_optuna_result = evaluate_classification_model(
    model_name="XGBoost Optuna",
    y_true=y_test,
    y_pred=xgb_optuna_pred,
    y_proba=xgb_optuna_proba
)

xgb_optuna_result

# 예상 출력:
# {'model_name': 'XGBoost Optuna',
#  'accuracy': 0.9736842105263158,
#  'precision_malignant': 1.0,
#  'recall_malignant': 0.9285714285714286,
#  'f1_malignant': 0.9629629629629629,
#  'roc_auc': 0.9923941798941799,
#  'TN': np.int64(72),
#  'FP': np.int64(0),
#  'FN': np.int64(3),
#  'TP': np.int64(39)}

{'model_name': 'XGBoost Optuna',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9940476190476191,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

### 4-4. XGBoost 3종 비교
Base · RandomizedSearchCV · Optuna 결과를 하나의 표로 비교합니다.

In [ ]:
# 세 결과를 하나의 표로 모읍니다.
xgb_tuning_results_df = pd.DataFrame([
    xgb_base_result,
    xgb_random_search_result,
    xgb_optuna_result
])

xgb_tuning_results_df

,model_name,accuracy,precision_malignant,recall_malignant,f1_malignant,roc_auc,TN,FP,FN,TP
0,XGBoost Base,0.964912,1.0,0.904762,0.950000,0.994378,72,0,4,38
1,XGBoost RandomizedSearchCV,0.973684,1.0,0.928571,0.962963,0.994048,72,0,3,39
2,XGBoost Optuna,0.964912,1.0,0.904762,0.950000,0.994048,72,0,4,38


## 5. 다른 모델 비교
Logistic Regression · RandomForest · LightGBM

### 5-1. Logistic Regression

In [ ]:
# 전처리(StandardScaler) + 모델(LogisticRegression)을 순서대로 묶습니다.
logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

# Pipeline 전체를 한 번에 학습 (scaler -> model 순서로 자동 처리)
logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_proba = logistic_model.predict_proba(X_test)[:, 1]

logistic_result = evaluate_classification_model(
    model_name="Logistic Regression",
    y_true=y_test,
    y_pred=logistic_pred,
    y_proba=logistic_proba
)
logistic_result

# 예상 출력:
# {'model_name': 'Logistic Regression',
#  'accuracy': 0.9649122807017544,
#  'precision_malignant': 0.975,
#  'recall_malignant': 0.9285714285714286,
#  'f1_malignant': 0.9512195121951219,
#  'roc_auc': 0.996031746031746,
#  'TN': np.int64(71),
#  'FP': np.int64(1),
#  'FN': np.int64(3),
#  'TP': np.int64(39)}

{'model_name': 'Logistic Regression',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 0.975,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9512195121951219,
 'roc_auc': 0.996031746031746,
 'TN': np.int64(71),
 'FP': np.int64(1),
 'FN': np.int64(3),
 'TP': np.int64(39)}

### 5-2. RandomForest

In [ ]:
# 트리 200개로 구성된 RandomForest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_result = evaluate_classification_model(
    model_name="RandomForest",
    y_true=y_test,
    y_pred=rf_pred,
    y_proba=rf_proba
)

rf_result

# 예상 출력:
# {'model_name': 'RandomForest',
#  'accuracy': 0.9649122807017544,
#  'precision_malignant': 1.0,
#  'recall_malignant': 0.9047619047619048,
#  'f1_malignant': 0.95,
#  'roc_auc': 0.994212962962963,
#  'TN': np.int64(72),
#  'FP': np.int64(0),
#  'FN': np.int64(4),
#  'TP': np.int64(38)}

{'model_name': 'RandomForest',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.994212962962963,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

### 5-3. LightGBM

In [ ]:
# LightGBM 모델 (verbose=-1로 학습 로그를 끔)
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]

lgbm_result = evaluate_classification_model(
    model_name="LightGBM",
    y_true=y_test,
    y_pred=lgbm_pred,
    y_proba=lgbm_proba
)

lgbm_result

# 예상 출력:
# {'model_name': 'LightGBM',
#  'accuracy': 0.9649122807017544,
#  'precision_malignant': 1.0,
#  'recall_malignant': 0.9047619047619048,
#  'f1_malignant': 0.95,
#  'roc_auc': 0.9947089947089948,
#  'TN': np.int64(72),
#  'FP': np.int64(0),
#  'FN': np.int64(4),
#  'TP': np.int64(38)}

{'model_name': 'LightGBM',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9947089947089948,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}